In [3]:
import pandas as pd
import numpy as np
import os

# 1. Define paths
input_path = '../../data/processed/df_main_clean.parquet'
if not os.path.exists(input_path):
    input_path = 'data/processed/df_main_clean.parquet'

print(f"🔄 Loading clean baseline dataset from: {input_path}")
df = pd.read_parquet(input_path)
print(f"✅ Initial shape: {df.shape[0]:,} rows | {df.shape[1]} columns")

🔄 Loading clean baseline dataset from: ../../data/processed/df_main_clean.parquet
✅ Initial shape: 307,511 rows | 23 columns


## 🟢 Cluster 1: Financial Leverage & Burden

### 💡 Business Justification:
In credit scoring, raw income or credit values alone fail to capture a borrower's actual debt capacity. This cluster constructs leverage and debt service ratios to measure the borrower's financial stress and monthly cash flow burden. High debt-to-income and annuity-to-income ratios directly correlate with higher default rates due to liquidity shocks.

In [4]:
# Debt-to-income leverage ratio
df['CREDIT_TO_INCOME'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
print(f"Added CREDIT_TO_INCOME | Mean: {df['CREDIT_TO_INCOME'].mean():.4f}")

Added CREDIT_TO_INCOME | Mean: 3.9579


In [5]:
# Monthly debt service burden ratio
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
print(f"Added ANNUITY_TO_INCOME | Mean: {df['ANNUITY_TO_INCOME'].mean():.4f}")

Added ANNUITY_TO_INCOME | Mean: 0.1809


In [6]:
# Estimated loan duration in months
df['CREDIT_TO_ANNUITY'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'] + 1e-5)
print(f"Added CREDIT_TO_ANNUITY | Mean: {df['CREDIT_TO_ANNUITY'].mean():.4f}")

Added CREDIT_TO_ANNUITY | Mean: 21.6124


In [7]:
# Disposable income per family member
if 'CNT_FAM_MEMBERS' in df.columns:
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1e-5)
    print(f"Added INCOME_PER_PERSON | Mean: {df['INCOME_PER_PERSON'].mean():.2f}")

Added INCOME_PER_PERSON | Mean: 92656.79


## 🔵 Cluster 2: Credit vs Goods Price (Purchase Financing & Down Payment)

### 💡 Business Justification:
When applicants request credit for purchasing consumer goods, the relationship between the requested loan amount (`AMT_CREDIT`) and the actual merchandise value (`AMT_GOODS_PRICE`) indicates borrower commitment. A significant down payment (`GOODS_CREDIT_DIFF > 0`) signals higher personal financial stake and lowers default risk (Loan-To-Value concept). Conversely, over-financing (`AMT_CREDIT > AMT_GOODS_PRICE`) suggests potential hidden fees or high-risk financing behavior.

In [8]:
# Loan-to-value (LTV) ratio for consumer purchases
if 'AMT_GOODS_PRICE' in df.columns:
    df['CREDIT_TO_GOODS_RATIO'] = df['AMT_CREDIT'] / (df['AMT_GOODS_PRICE'] + 1e-5)
    print(f"Added CREDIT_TO_GOODS_RATIO | Mean: {df['CREDIT_TO_GOODS_RATIO'].mean():.4f}")

Added CREDIT_TO_GOODS_RATIO | Mean: 1.1229


In [9]:
# Estimated down payment (personal equity contribution)
if 'AMT_GOODS_PRICE' in df.columns:
    df['GOODS_CREDIT_DIFF'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    print(f"Added GOODS_CREDIT_DIFF | Mean: {df['GOODS_CREDIT_DIFF'].mean():.2f}")

Added GOODS_CREDIT_DIFF | Mean: -60863.72


In [10]:
# Monthly installment burden relative to merchandise value
if 'AMT_GOODS_PRICE' in df.columns:
    df['ANNUITY_TO_GOODS_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_GOODS_PRICE'] + 1e-5)
    print(f"Added ANNUITY_TO_GOODS_RATIO | Mean: {df['ANNUITY_TO_GOODS_RATIO'].mean():.4f}")

Added ANNUITY_TO_GOODS_RATIO | Mean: 0.0599


## 🟡 Cluster 3: Demographics, Employment & Vehicle Age

### 💡 Business Justification:
Employment stability and career progression are major pillars of creditworthiness. This cluster handles the critical data anomaly in `DAYS_EMPLOYED` (value `365243`, representing unemployed/pensioners) and standardizes age and employment tenure into positive year units. Furthermore, features like `EMPLOYED_TO_AGE_RATIO` capture lifetime career stability, while `CAR_TO_BIRTH_RATIO` assesses vehicle depreciation relative to the applicant's life stage.

In [11]:
# Flag the 365243 anomaly (unemployed/pensioners) and replace with NaN
if 'DAYS_EMPLOYED' in df.columns:
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED_CLEAN'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    print(f"Handled DAYS_EMPLOYED anomaly | Anomaly count: {df['DAYS_EMPLOYED_ANOM'].sum():,}")

Handled DAYS_EMPLOYED anomaly | Anomaly count: 0


In [12]:
# Applicant age in positive years
if 'DAYS_BIRTH' in df.columns:
    df['AGE_YEARS'] = df['DAYS_BIRTH'].abs() / 365.25
    print(f"Added AGE_YEARS | Mean: {df['AGE_YEARS'].mean():.2f} years")

Added AGE_YEARS | Mean: 43.91 years


In [13]:
# Employment tenure in positive years
if 'DAYS_EMPLOYED_CLEAN' in df.columns:
    df['EMPLOYED_YEARS'] = df['DAYS_EMPLOYED_CLEAN'].abs() / 365.25
    print(f"Added EMPLOYED_YEARS | Mean: {df['EMPLOYED_YEARS'].mean():.2f} years")

Added EMPLOYED_YEARS | Mean: 6.53 years


In [14]:
# Proportion of lifetime spent in active employment
if 'EMPLOYED_YEARS' in df.columns and 'AGE_YEARS' in df.columns:
    df['EMPLOYED_TO_AGE_RATIO'] = df['EMPLOYED_YEARS'] / (df['AGE_YEARS'] + 1e-5)
    print(f"Added EMPLOYED_TO_AGE_RATIO | Mean: {df['EMPLOYED_TO_AGE_RATIO'].mean():.4f}")

Added EMPLOYED_TO_AGE_RATIO | Mean: 0.1569


In [15]:
# Career earning trajectory per year of employment
if 'EMPLOYED_YEARS' in df.columns and 'AMT_INCOME_TOTAL' in df.columns:
    df['INCOME_PER_EMPLOYED_YEAR'] = df['AMT_INCOME_TOTAL'] / (df['EMPLOYED_YEARS'] + 1e-5)
    print(f"Added INCOME_PER_EMPLOYED_YEAR | Mean: {df['INCOME_PER_EMPLOYED_YEAR'].mean():.2f}")

Added INCOME_PER_EMPLOYED_YEAR | Mean: 197814.91


## 🟣 Cluster 4: External Credit Scores Aggregation

### 💡 Business Justification:
The external credit scores (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`) are the strongest individual predictors in the Home Credit dataset. Creating row-wise aggregations summarizes consensus among independent credit bureaus (`EXT_SOURCE_MEAN`), quantifies inter-bureau disagreement (`EXT_SOURCE_STD`), and imposes a severe penalty if any single rating agency flags high risk (`EXT_SOURCE_PROD`).

In [16]:
# Mean score across external bureaus
ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df.columns]
if len(ext_cols) > 0:
    df['EXT_SOURCE_MEAN'] = df[ext_cols].mean(axis=1)
    print(f"Added EXT_SOURCE_MEAN | Mean: {df['EXT_SOURCE_MEAN'].mean():.4f}")

Added EXT_SOURCE_MEAN | Mean: 0.5093


In [17]:
# Standard deviation among external bureau scores
if len(ext_cols) > 0:
    df['EXT_SOURCE_STD'] = df[ext_cols].std(axis=1).fillna(0)
    print(f"Added EXT_SOURCE_STD | Mean: {df['EXT_SOURCE_STD'].mean():.4f}")

Added EXT_SOURCE_STD | Mean: 0.1331


In [18]:
# Product interaction of external bureau scores
if len(ext_cols) > 0:
    df['EXT_SOURCE_PROD'] = df[ext_cols].prod(axis=1)
    print(f"Added EXT_SOURCE_PROD | Mean: {df['EXT_SOURCE_PROD'].mean():.4f}")

Added EXT_SOURCE_PROD | Mean: 0.2502


## 🟠 Cluster 5: Social Circle Risk Ratios

### 💡 Business Justification:
Social circle features capture environmental credit contagion risk. By computing the proportion of default observations relative to total social circle observations over 30-day and 60-day windows, the model measures the default density in the applicant's immediate social surroundings.

In [19]:
# Default ratio in 30-day social circle observation window
if 'DEF_30_CNT_SOCIAL_CIRCLE' in df.columns and 'OBS_30_CNT_SOCIAL_CIRCLE' in df.columns:
    df['DEF_TO_OBS_30'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] / (df['OBS_30_CNT_SOCIAL_CIRCLE'] + 1e-5)
    print(f"Added DEF_TO_OBS_30 | Mean: {df['DEF_TO_OBS_30'].mean():.4f}")

Added DEF_TO_OBS_30 | Mean: 0.0668


In [20]:
# Default ratio in 60-day social circle observation window
if 'DEF_60_CNT_SOCIAL_CIRCLE' in df.columns and 'OBS_60_CNT_SOCIAL_CIRCLE' in df.columns:
    df['DEF_TO_OBS_60'] = df['DEF_60_CNT_SOCIAL_CIRCLE'] / (df['OBS_60_CNT_SOCIAL_CIRCLE'] + 1e-5)
    print(f"Added DEF_TO_OBS_60 | Mean: {df['DEF_TO_OBS_60'].mean():.4f}")

Added DEF_TO_OBS_60 | Mean: 0.0503


## 🔴 Cluster 6: Group Aggregation on Income (Peer Segment Comparison)

### 💡 Business Justification:
Absolute income level lacks context without peer benchmarks. Group aggregation evaluates an applicant's relative financial standing by benchmarking their income against the mean income of their specific categorical segment (`OCCUPATION_TYPE`, `NAME_EDUCATION_TYPE`, and `NAME_INCOME_TYPE`). This allows tree models to identify high performers or underperformers within their peer socio-economic group.

In [21]:
# Income benchmark by Occupation
if 'OCCUPATION_TYPE' in df.columns:
    occ_mean = df.groupby('OCCUPATION_TYPE')['AMT_INCOME_TOTAL'].transform('mean')
    df['INCOME_RATIO_BY_OCCUPATION'] = df['AMT_INCOME_TOTAL'] / (occ_mean + 1e-5)
    df['INCOME_DIFF_BY_OCCUPATION'] = df['AMT_INCOME_TOTAL'] - occ_mean
    print(f"Added Occupation Income Aggregations | Ratio Mean: {df['INCOME_RATIO_BY_OCCUPATION'].mean():.4f}")

Added Occupation Income Aggregations | Ratio Mean: 1.0000


In [22]:
# Income benchmark by Education Level
if 'NAME_EDUCATION_TYPE' in df.columns:
    edu_mean = df.groupby('NAME_EDUCATION_TYPE')['AMT_INCOME_TOTAL'].transform('mean')
    df['INCOME_RATIO_BY_EDUCATION'] = df['AMT_INCOME_TOTAL'] / (edu_mean + 1e-5)
    df['INCOME_DIFF_BY_EDUCATION'] = df['AMT_INCOME_TOTAL'] - edu_mean
    print(f"Added Education Income Aggregations | Ratio Mean: {df['INCOME_RATIO_BY_EDUCATION'].mean():.4f}")

Added Education Income Aggregations | Ratio Mean: 1.0000


In [23]:
# Income benchmark by Income Source Type
if 'NAME_INCOME_TYPE' in df.columns:
    inc_type_mean = df.groupby('NAME_INCOME_TYPE')['AMT_INCOME_TOTAL'].transform('mean')
    df['INCOME_RATIO_BY_INCOMETYPE'] = df['AMT_INCOME_TOTAL'] / (inc_type_mean + 1e-5)
    df['INCOME_DIFF_BY_INCOMETYPE'] = df['AMT_INCOME_TOTAL'] - inc_type_mean
    print(f"Added Income Type Aggregations | Ratio Mean: {df['INCOME_RATIO_BY_INCOMETYPE'].mean():.4f}")

Added Income Type Aggregations | Ratio Mean: 1.0000


In [24]:
# Export the feature-engineered dataset to Parquet format
import os
output_path = '/data/processed/table/df_main_clean_fe.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_parquet(output_path, index=False)

print("=" * 60)
print(f"🎉 SPRINT 2 FEATURE ENGINEERING COMPLETE!")
print(f"📊 Final Dataset Shape: {df.shape[0]:,} rows | {df.shape[1]} columns")
print(f"💾 Saved successfully to: {output_path}")
print("=" * 60)

🎉 SPRINT 2 FEATURE ENGINEERING COMPLETE!
📊 Final Dataset Shape: 307,511 rows | 47 columns
💾 Saved successfully to: /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_fe.parquet
